# Lab 02: Webhook Integration - Build the Server

**Lab**: 02-webhook-integration  
**Duration**: ~5 minutes  
**Prerequisites**: Completed `01_webhook_fundamentals.ipynb`, Python 3.13+

## Learning Objectives

By the end of this notebook, you will:
- Understand the webhook server code structure
- Know how to implement signature verification
- Handle different event types

---

## Setup

First, let's verify our environment is ready.

<!-- PRESENTER: Ensure attendees have Python and dependencies installed -->

In [ ]:
# Verify Python version
import sys
print(f"Python version: {sys.version}")

# Verify required packages
try:
    import stripe
    import flask
    import dotenv
    print(f"stripe: {stripe.VERSION}")
    print(f"flask: {flask.__version__}")
    print("All packages installed!")
except ImportError as e:
    print(f"Missing package: {e}")
    print("Run: pip install stripe flask python-dotenv")

## Server Architecture

Our webhook server has a simple architecture:

```
+------------------+
|   Flask App      |
+------------------+
         |
         v
+------------------+
| POST /webhook    |  <-- Single endpoint
+------------------+
         |
         v
+------------------+
| 1. Parse JSON    |
| 2. Verify Sig    |
| 3. Route Event   |
| 4. Handle Event  |
| 5. Return 200    |
+------------------+
```

## Step 1: Environment Setup

Let's look at how the server loads configuration.

<!-- PRESENTER: Explain why we use environment variables for secrets -->

### Installing Stripe CLI

If not installed, follow the [installation guide](https://stripe.com/docs/stripe-cli#install):

```bash
# macOS (Homebrew)
brew install stripe/stripe-cli/stripe

# Windows (Scoop)
scoop install stripe

# Linux
# Download from: https://github.com/stripe/stripe-cli/releases

In [10]:
# This is how the server loads environment variables
import os
from dotenv import load_dotenv

dotenv_path = ('.env')
load_dotenv(dotenv_path)

# Get configuration
STRIPE_SECRET_KEY = os.environ.get('STRIPE_SECRET_KEY')

#Get the webhook secret from `stripe listen` when testing locally
STRIPE_WEBHOOK_SECRET = os.environ.get('STRIPE_WEBHOOK_SECRET')

print(f"Secret Key: {'Set' if STRIPE_SECRET_KEY and len(STRIPE_SECRET_KEY) > 10 else 'NOT SET'}")
print(f"Webhook Secret: {'Set' if STRIPE_WEBHOOK_SECRET and len(STRIPE_WEBHOOK_SECRET) > 10 else 'NOT SET'}")

Secret Key: Set
Webhook Secret: Set


### The .env File

Your `.env` file should contain:

```bash
STRIPE_SECRET_KEY=sk_test_...
STRIPE_WEBHOOK_SECRET=whsec_...
```

**Note**: Get the webhook secret from `stripe listen` when testing locally.

## Step 2: Flask App Structure

Let's examine the server code piece by piece.

<!-- PRESENTER: Walk through each section of the code -->

In [ ]:
# Display the server code
!cat src/server.py

## Step 3: Understanding the Webhook Handler

Let's break down the key parts of the webhook handler:

### Part 1: Parse the Request

```python
@app.route('/webhook', methods=['POST'])
def webhook():
    payload = request.data  # Raw request body
    
    # First, try to parse as JSON
    try:
        event = json.loads(payload)
    except json.decoder.JSONDecodeError as e:
        return jsonify(success=False), 400
```

**Key Point**: We need the raw payload for signature verification.

### Part 2: Verify the Signature

```python
if endpoint_secret:
    sig_header = request.headers.get('stripe-signature')
    try:
        event = stripe.Webhook.construct_event(
            payload, sig_header, endpoint_secret
        )
    except stripe.error.SignatureVerificationError as e:
        print('Webhook signature verification failed.')
        return jsonify(success=False), 400
```

**Key Points**:
- `construct_event()` verifies AND parses the event
- If verification fails, reject the request immediately
- The webhook secret (`whsec_...`) is unique per endpoint

### Part 3: Route by Event Type

```python
if event['type'] == 'payment_intent.succeeded':
    payment_intent = event['data']['object']
    print('Payment for {} succeeded'.format(payment_intent['amount']))
    # handle_payment_intent_succeeded(payment_intent)
    
elif event['type'] == 'payment_method.attached':
    payment_method = event['data']['object']
    # handle_payment_method_attached(payment_method)
    
else:
    print('Unhandled event type {}'.format(event['type']))
```

**Key Points**:
- Use `event['type']` to route to the right handler
- `event['data']['object']` contains the actual Stripe object
- Always handle the "else" case for unexpected events

### Part 4: Return Success

```python
return jsonify(success=True)
```

**Important**: Always return `200 OK` to acknowledge receipt. Stripe will retry if you return an error.

## Step 4: Adding New Event Handlers

To handle a new event type, add an `elif` block:

```python
elif event['type'] == 'customer.subscription.created':
    subscription = event['data']['object']
    customer_id = subscription['customer']
    plan_id = subscription['items']['data'][0]['price']['id']
    
    # Your business logic
    activate_subscription(customer_id, plan_id)
```

## Summary

In this notebook, you learned:

- **Environment setup**: Using `.env` for secrets
- **Server structure**: Single `/webhook` endpoint
- **Signature verification**: Using `construct_event()`
- **Event routing**: Using `event['type']` to dispatch

## Next Steps

Continue to `03_test_webhooks.ipynb` to test the server with Stripe CLI!